# Preprocessing

Before matching, raw choices need to be validated: project codes must exist, projects must be open to the student's course, and duplicate picks of the same project need to be collapsed to the best rank. This notebook applies `matched.preprocess` to the data introduced in [The data](01-data.ipynb).

We start by reproducing the loading and dict-construction steps from the previous notebook:

In [ ]:
import pathlib

import pandas as pd

import matched

raw = pathlib.Path("./raw")

projects = pd.read_csv(raw / "projects.csv").set_index("code")
choices_wide = pd.read_csv(raw / "internal-choices.csv").set_index("username")
choice_cols = [c for c in choices_wide.columns if c.startswith("choice")]

choices = {
    username: [code for code in row[choice_cols] if pd.notna(code)]
    for username, row in choices_wide.iterrows()
}
scores = choices_wide.score.to_dict()
courses = choices_wide.course.to_dict()
nmax = projects.nmax.to_dict()
eligible_courses = {
    code: [c for c in ("course1", "course2") if row[c]]
    for code, row in projects.iterrows()
}

sum(len(codes) for codes in choices.values())

### Invalid project codes

`filter_invalid_code` drops codes that aren't a key in `nmax` (i.e. not a known project) - for example a typo, or a project withdrawn after the form was opened. We check what changes for each student:

In [ ]:
before = choices
choices = matched.filter_invalid_code(choices, nmax)

{
    username: before[username]
    for username in choices
    if before[username] != choices[username]
}

Nothing is dropped here - every code in this dataset is a known project. In practice, this step guards against typos or a project withdrawn after the form was opened.

### Projects not offered to the student's course

`filter_invalid_course` drops a code if the project isn't eligible for the student's course (checked against `eligible_courses`).

In [ ]:
before = choices
choices = matched.filter_invalid_course(choices, courses, eligible_courses)

{
    username: (before[username], choices[username])
    for username in choices
    if before[username] != choices[username]
}

Three students lose a choice. `kl112`, for example, is on `course2` but had picked `alfa-012` - a project only eligible for `course1` - as their first choice; after preprocessing, their first *valid* choice becomes whatever they ranked next. This matters for the outcome: in [The algorithm](03-algorithm.ipynb) we'll see `kl112` allocated their second choice rather than the (invalid) first one.

### Duplicate picks

`deduplicate` collapses repeated `(username, code)` picks, keeping only the best (lowest) `choice` rank - e.g. if a student accidentally listed the same project twice.

In [ ]:
before = choices
choices = matched.deduplicate(choices)

(
    sum(len(codes) for codes in before.values()),
    sum(len(codes) for codes in choices.values()),
)

No duplicates in this dataset, so nothing changes here - but the same pipeline handles it automatically when they do occur.

### Putting it together

The three steps compose cleanly by sequential reassignment, in this order - code validity, then course validity, then de-duplication:

In [ ]:
raw_choices = {
    username: [code for code in row[choice_cols] if pd.notna(code)]
    for username, row in choices_wide.iterrows()
}

clean_choices = matched.filter_invalid_code(raw_choices, nmax)
clean_choices = matched.filter_invalid_course(clean_choices, courses, eligible_courses)
clean_choices = matched.deduplicate(clean_choices)

clean_choices["tb241"], clean_choices["kl112"]

`clean_choices` (along with `scores` and `nmax`) is now ready to allocate. [The algorithm](03-algorithm.ipynb) walks through how `match` turns this into an allocation, and what happens when a project is oversubscribed.